In [ ]:
import math
import random

In [2]:
#%pip install graphviz
#%pip install torch
#%pip install --upgrade pip

In [3]:

# brew install graphviz

from graphviz import Digraph

In [4]:
class Value:
    def __init__(self, data, _children=(), _op = '', label = ''):
        self.data = data
        self.grad = 0.0
        
        self.addgrad = 0.0
        
        self._backward = lambda:None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.addgrad += out.grad
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
            
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.addgrad += other.data * out.grad
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data **other, (self,), f'**(other)')

        def _backward():
            self.addgrad += other * (self.data**(other-1)) * out.grad
            self.grad += other * (self.data**(other-1)) * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.addgrad += (1 - t**2) * out.grad
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.addgrad += out.data * out.grad
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        topo
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()






In [5]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))

        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad ), shape='record')
        if n._op:
            dot.node(name = uid + n._op, label = n._op)
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot
    

In [9]:
class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        # w * x + b
        act = sum((wi*xi for wi, xi in zip(self.w,x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        #here as you can see we are only collecting weights and biases
        #Hence when we update we only update weights and biases according to their current gradient
        return self.w + [self.b]


class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
        
        # params = []
        # for neuron in self.neurons:
        #     p = neuron.parameters()
        #     params.extend(p)
        # return params


class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range (len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
        




# nin = 2 means it creates two weights. So basically Neuron(2) is just one neuron we are introducing that does multiplication of the 
# two weights with two inputs

# x in __call__ function is inputs [2.0, 3.0]
# nout is the number of neurones in a layer

In [10]:

n = MLP(3, [4,4,1])



In [11]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0]


In [12]:

for k in range(10):
    
    #forward pass
    ypred = [n(x) for x in xs]

    #loss calculation
    loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)], Value(0.0))

    #Flush the grads to avoid accumulation at every backward pass
    for p in n.parameters():
        p.grad = 0.0


    #backward pass
    loss.backward()

    #update weights and biases only
    for p in n.parameters():
        p.data += -0.05 * p.grad



    print(k, loss.data)
    
    

0 4.9838734905080075
1 2.550797313096659
2 1.438829487448769
3 0.8163911289854635
4 0.5008159576764504
5 0.34243528020482633
6 0.25407090286324396
7 0.19951012453951544
8 0.16307026612250192
9 0.13725337243076638


In [13]:
ypred

[Value(data=0.854389182226256),
 Value(data=-0.8762666834082959),
 Value(data=-0.7734497587820727),
 Value(data=0.7777030887588764)]